# Notebook 07: ResNet from Scratch

## Why This Matters

ResNet is not just a CIFAR-10 benchmark -- it is the foundational architecture that unlocked training arbitrarily deep networks and that introduced the shortcut/skip connection pattern now ubiquitous in transformers, U-Nets, and virtually every modern architecture. Understanding why residual connections work (mathematically and intuitively), and being able to build one from scratch with correct Conv-BN-ReLU ordering, BatchNorm parameter interaction, and projection shortcuts, is a litmus test of practical PyTorch depth.

## Background

The central problem ResNet (He et al., 2015) solved: deeper networks were performing WORSE than shallower ones -- not because of overfitting, but because optimisation itself failed. A 56-layer plain network had higher training error than a 20-layer one.

The key insight: if the optimal function for a layer block is the identity, it is easier to learn $H(x) = F(x) + x$ where $F(x) \to 0$ (residual close to zero) than to learn the identity directly as $H(x) = x$. By construction, if $F(x) = 0$ the block becomes an identity mapping.

$$H(x) = F(x) + x$$

The skip connection (shortcut) lets gradients flow directly through the addition node during backprop, bypassing the convolutional layers entirely:

In [ ]:
%pip install -q torch numpy matplotlib torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print("Ready.")

## 1. Why bias=False Before BatchNorm

This is one of the most asked questions about ResNet implementations. BatchNorm normalises each channel to zero mean and unit variance, then applies a learnable `gamma` (scale) and `beta` (shift). If a Conv layer has a bias, that bias is immediately cancelled by BatchNorm's mean-subtraction step -- it has zero effect on the output. Adding it wastes memory and adds parameters that contribute nothing. The convention is: `Conv2d(..., bias=False)` when followed by BatchNorm.

In [ ]:
# Demonstrate that conv bias is irrelevant before BatchNorm
torch.manual_seed(0)
x = torch.randn(4, 3, 8, 8)

# With bias
conv_with_bias    = nn.Conv2d(3, 8, 3, padding=1, bias=True)
# Without bias (same init for weight)
conv_without_bias = nn.Conv2d(3, 8, 3, padding=1, bias=False)
conv_without_bias.weight.data.copy_(conv_with_bias.weight.data)

bn = nn.BatchNorm2d(8)

out_with    = bn(conv_with_bias(x))
out_without = bn(conv_without_bias(x))

print(f"Output with bias:    mean={out_with.mean().item():.6f}  std={out_with.std().item():.4f}")
print(f"Output without bias: mean={out_without.mean().item():.6f}  std={out_without.std().item():.4f}")
print(f"Max diff:            {(out_with - out_without).abs().max().item():.6f}")
print("\nConclusion: bias makes no difference after BN. Skip it to save memory.")

## 2. BasicBlock -- The Core Residual Unit

The BasicBlock is the fundamental unit of ResNet-18 and ResNet-34. It consists of:
1. Conv 3x3 -> BN -> ReLU
2. Conv 3x3 -> BN
3. Shortcut (identity or 1x1 conv projection for dimension matching)
4. Addition + final ReLU

The 1x1 projection shortcut is needed when the number of channels or spatial resolution changes (stride > 1). Without it, the addition would fail due to shape mismatch.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1  # used by larger ResNets to track channel widths

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Main path: two 3x3 convolutions
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # Shortcut: project if dimensions change
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        # Main path
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # Residual addition
        out = out + self.shortcut(x)
        return F.relu(out)

# Test the block
block = BasicBlock(16, 32, stride=2)
x_test = torch.randn(4, 16, 8, 8)
out_test = block(x_test)
print(f"BasicBlock: input {x_test.shape} -> output {out_test.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

# Gradient flows through both paths
loss = out_test.sum()
loss.backward()
print(f"Gradient norms -- conv1: {block.conv1.weight.grad.norm():.4f}  "
      f"shortcut: {block.shortcut[0].weight.grad.norm():.4f}")

## 3. AdaptiveAvgPool2d for Input-Size Agnosticism

Classic ResNets end with a `GlobalAveragePooling` that collapses spatial dimensions to 1x1. `AdaptiveAvgPool2d((1, 1))` achieves this for any input spatial size, making the classifier head independent of the input resolution. Without this, a model trained on 32x32 inputs would fail on 224x224 inputs.

In [ ]:
# Demonstrate input-size agnosticism
backbone = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),  # always outputs (B, C, 1, 1)
)

for h, w in [(32, 32), (64, 64), (224, 224), (100, 150)]:
    x_test = torch.randn(2, 3, h, w)
    out = backbone(x_test)
    print(f"Input ({h}x{w}) -> output {out.shape}")

print("\nAdaptiveAvgPool2d always produces (B, C, 1, 1) regardless of input size")

## 4. SmallResNet for CIFAR-10

We build a compact ResNet designed for 32x32 CIFAR-10 inputs. The standard ImageNet ResNet uses a 7x7 conv with stride 2 followed by max pooling, which is too aggressive for small images. For CIFAR-10 we use a 3x3 conv with no pooling at the stem, then gradually reduce spatial resolution through strided convolutions in the residual blocks.

In [ ]:
class SmallResNet(nn.Module):
    def __init__(self, num_classes=10, base_channels=32):
        super().__init__()
        c = base_channels

        # Stem: 3x3 conv, no downsampling yet
        self.stem = nn.Sequential(
            nn.Conv2d(3, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(),
        )

        # Stage 1: (B, c, 32, 32) -> (B, c, 32, 32)
        self.stage1 = nn.Sequential(
            BasicBlock(c, c, stride=1),
            BasicBlock(c, c, stride=1),
        )

        # Stage 2: (B, c, 32, 32) -> (B, 2c, 16, 16)
        self.stage2 = nn.Sequential(
            BasicBlock(c, 2*c, stride=2),
            BasicBlock(2*c, 2*c, stride=1),
        )

        # Stage 3: (B, 2c, 16, 16) -> (B, 4c, 8, 8)
        self.stage3 = nn.Sequential(
            BasicBlock(2*c, 4*c, stride=2),
            BasicBlock(4*c, 4*c, stride=1),
        )

        # Head
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(4*c, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.pool(x).flatten(1)  # (B, 4c)
        return self.classifier(x)

model = SmallResNet(num_classes=10, base_channels=32).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SmallResNet parameters: {total_params:,}")

x_test = torch.randn(4, 3, 32, 32, device=device)
out_test = model(x_test)
print(f"Output shape: {out_test.shape}")

# Verify gradient flow through all stages
loss = out_test.sum()
loss.backward()
print("\nGradient norms per stage:")
for name, p in model.named_parameters():
    if p.grad is not None and "weight" in name:
        print(f"  {name:45s}: {p.grad.norm():.4f}")

## 5. Full Training Loop with Cosine Schedule and Mixed Precision

Putting it all together: training SmallResNet on synthetic CIFAR-10-like data with BF16/FP16 autocast and a cosine LR schedule.

In [ ]:
import math
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Synthetic CIFAR-10-like dataset
N_TRAIN, N_VAL = 512, 128
X_train = torch.randn(N_TRAIN, 3, 32, 32)
y_train = torch.randint(0, 10, (N_TRAIN,))
X_val   = torch.randn(N_VAL,   3, 32, 32)
y_val   = torch.randint(0, 10, (N_VAL,))

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=64)

model_train = SmallResNet(num_classes=10, base_channels=32).to(device)
optimizer_train = optim.SGD(model_train.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler_train = optim.lr_scheduler.CosineAnnealingLR(optimizer_train, T_max=5)
criterion = nn.CrossEntropyLoss()

amp_device_type = "cuda" if device == "cuda" else "cpu"
amp_dtype = torch.bfloat16

train_losses, val_accs = [], []

for epoch in range(5):
    # Training
    model_train.train()
    epoch_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer_train.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype):
            logits = model_train(X_b)
            loss = criterion(logits, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model_train.parameters(), max_norm=1.0)
        optimizer_train.step()
        epoch_loss += loss.item()
    scheduler_train.step()

    # Validation
    model_train.eval()
    correct = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model_train(X_b).argmax(1)
            correct += (preds == y_b).sum().item()
    val_acc = correct / N_VAL

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_accs.append(val_acc)
    print(f"epoch {epoch+1}:  loss={avg_loss:.4f}  val_acc={val_acc:.3f}  lr={scheduler_train.get_last_lr()[0]:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(train_losses, marker="o"); ax1.set_title("Training Loss"); ax1.set_xlabel("epoch")
ax2.plot(val_accs, marker="o", color="orange"); ax2.set_title("Val Accuracy"); ax2.set_xlabel("epoch")
ax2.axhline(0.1, linestyle="--", color="gray", label="random chance"); ax2.legend()
plt.tight_layout(); plt.savefig("/tmp/resnet_training.png", dpi=80); plt.show()
print("\nTraining complete.")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| Residual connection | H(x) = F(x) + x; easier to learn near-zero residual than identity |
| bias=False before BN | BN cancels the bias; it has zero effect and wastes memory |
| Projection shortcut | 1x1 conv to match channels/stride when dimensions change |
| AdaptiveAvgPool2d | Collapses any spatial size to (1,1); makes head size-agnostic |
| CIFAR stem vs ImageNet stem | 3x3 conv + no pool for small images; 7x7 + maxpool for ImageNet |
| BasicBlock vs Bottleneck | BasicBlock: two 3x3 convs; Bottleneck: 1x1-3x3-1x1 for deeper nets |

### Common Pitfalls
- Using `bias=True` with BatchNorm -- wasted parameters, no benefit
- Applying ReLU before the residual addition -- harms gradient flow
- Using MaxPool2d(3, stride=2) on CIFAR-10 -- collapses spatial info too aggressively
- Forgetting the shortcut projection when channels change -- RuntimeError on addition
- Not setting `model.eval()` before validation -- BatchNorm uses batch stats instead of running stats

### Next: Notebook 08 -- torch.compile